# Hyperparameter Tuning

Use KerasTuner's Hyperband to automate hyperparameter optimization instead of manually tuning parameters. Search for the best combination of learning rate, number of filters, number of dense units, label smoothing rate, regularization strength, and dropout rate.

Takes around ~6 hours to run on Google Colab's GPU.

## Import Libraries

In [ ]:
from tensorflow import keras
import keras_tuner as kt
from tensorflow.keras import layers, models, optimizers, regularizers
from tensorflow.keras.callbacks import EarlyStopping
import pickle
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

## Load Preprocessed Data

In [ ]:
data = np.load('../data/processed_data.npz')

X_train = data['X_train']
X_test = data['X_test']
X_val = data['X_val']

X_train_normalized = data['X_train_normalized']
X_test_normalized = data['X_test_normalized']
X_val_normalized = data['X_val_normalized']

y_train = data['y_train']
y_test = data['y_test']
y_val = data['y_val']

y_train_cat = data['y_train_cat']
y_test_cat = data['y_test_cat']
y_val_cat = data['y_val_cat']

## Utilities

### Data Augmentation

In [ ]:
# layer that randomly applies transformations (flips, rotations, zooms) to each input image during training only
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

### Class Weights

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(enumerate(class_weights))

### Learning Rate Scheduler

In [ ]:
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

### Helper Functions

In [ ]:
# plots accuracy and loss training curves
def plot_training_curves(history, title):
    plt.figure(figsize=(12, 5))

    # accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history['accuracy'], label='train')
    plt.plot(history['val_accuracy'], label='val')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy');
    plt.legend();
    
    # loss
    plt.subplot(1, 2, 2)
    plt.plot(history['loss'], label='train')
    plt.plot(history['val_loss'], label='val')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend();

    plt.show()

emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']    

In [ ]:
# evaluates model, test accuracy & loss, confusion matrix, classification report
def evaluate_model(model, X_test, y_test, title):
    # test results
    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f'{title} - Test Accuracy: {test_accuracy:.4f} Test Loss: {test_loss:.4f}')
    
    # confusion matrix
    y_pred = np.argmax(model.predict(X_test), axis=1)
    y_true = np.argmax(y_test, axis=1)
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=emotion_labels).plot(cmap='Blues', xticks_rotation='vertical')
    plt.title(f'{title} - Confusion Matrix')
    plt.show()
    
    # classification report
    print(classification_report(y_true, y_pred, target_names=emotion_labels))

## Tuning

In [ ]:
def build_model(hp):
    filters_1 = hp.Choice('filters_1', [32, 64])
    filters_2 = hp.Choice('filters_2', [64, 128])
    filters_3 = hp.Choice('filters_3', [128, 256])
    filters_4 = hp.Choice('filters_4', [128, 256])
    dense_units = hp.Choice('dense_units', [128, 256])

    l2_strength = hp.Choice('l2_strength', [1e-5, 1e-4, 5e-4])
    dropout = hp.Choice('dropout', [0.25, 0.3, 0.35])
    dense_dropout = hp.Choice('dense_dropout', [0.4, 0.5])
    label_smoothing = hp.Choice('label_smoothing', [0.0, 0.05, 0.1])
    learning_rate = hp.Choice('learning_rate', [5e-4, 1e-4, 1e-3])

    use_block_4 = hp.Boolean('use_block_4')

    model = keras.Sequential([
        layers.Input(shape=(48, 48, 1)),

        data_augmentation,
        layers.Rescaling(1./255),

        # convolutional block 1
        layers.Conv2D(filters=filters_1,
                      kernel_size=(3, 3),
                      kernel_regularizer=regularizers.l2(l2_strength),
                      activation='relu',
                      padding='same'
        ),
        layers.BatchNormalization(),
        layers.Conv2D(filters=filters_1,
                      kernel_size=(3, 3),
                      kernel_regularizer=regularizers.l2(l2_strength),
                      activation='relu',
                      padding='same'
        ),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(dropout),

        # convolutional block 2
        layers.Conv2D(filters=filters_2,
                      kernel_size=(3, 3),
                      kernel_regularizer=regularizers.l2(l2_strength),
                      activation='relu',
                      padding='same'

        ),
        layers.BatchNormalization(),
        layers.Conv2D(filters=filters_2,
                     kernel_size=(3, 3),
                     kernel_regularizer=regularizers.l2(l2_strength),
                     activation='relu',
                     padding='same'
        ),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(dropout),

        # convolutional block 3
        layers.Conv2D(filters=filters_3,
                      kernel_size=(3, 3),
                      kernel_regularizer=regularizers.l2(l2_strength),
                      activation='relu',
                      padding='same'
        ),
        layers.BatchNormalization(),
        layers.Conv2D(filters=filters_3,
                      kernel_size=(3, 3),
                      kernel_regularizer=regularizers.l2(l2_strength),
                      activation='relu',
                      padding='same'
        ),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(dropout),

    ])

    # optional 4th convolutional block
    if use_block_4:
      model.add(layers.Conv2D(filters=filters_4,
                    kernel_size=(3, 3),
                    kernel_regularizer=regularizers.l2(l2_strength),
                    activation='relu',
                    padding='same'
      ))
      model.add(layers.BatchNormalization())
      model.add(layers.Conv2D(filters=filters_4,
                    kernel_size=(3, 3),
                    kernel_regularizer=regularizers.l2(l2_strength),
                    activation='relu',
                    padding='same'
      ))
      model.add(layers.BatchNormalization())
      model.add(layers.MaxPooling2D((2, 2)))
      model.add(layers.Dropout(dropout))

    # global pooling + dense layer
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(units=dense_units,
                  kernel_regularizer=regularizers.l2(l2_strength),
                  activation='relu'
    ))
    model.add(layers.Dropout(dense_dropout))
    model.add(layers.Dense(7, activation='softmax'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics=['accuracy']
    )

    return model

In [ ]:
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=50,
    factor=3,
    directory='../tuning',
    project_name='emotion_tuning',
    overwrite=False
)

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

tuner.search(
    X_train, y_train_cat,
    epochs=50,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best hyperparameters: ", best_hp.values)

best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

best_model.save('../models/tuned_model.keras')

In [ ]:
history_best = best_model.fit(
    X_train, y_train_cat,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop, reduce_lr]
)

with open('../history/history_best.pkl', 'wb') as f:
  pickle.dump(history_best.history, f)

In [ ]:
plot_training_curves(history_best, 'Best Model')
evaluate_model(model_best, X_test, y_test_cat, 'Best Model')